# 🚀 Модуль 17. Урок 1. Live-coding собеседование: SQL и Python скрининг 2027

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_17_career/lesson_01_sql_python_livecoding_interview.ipynb)

## 📌 Бизнес-контекст и методология
На техническом скрининге в продуктовые компании уровня Tier-1 кандидатов проверяют на умение решать нестандартные задачи без циклов:
1. **Gaps and Islands (Острова и разрывы):** поиск непрерывных цепочек активности (streak) пользователей за $O(N)$ времени.
2. **Time-to-Second-Event:** расчет задержки до повторной покупки на уровне когорт.
3. **Оконные агрегаты:** скользящее среднее и накопительный LTV с граничными условиями.

In [ ]:
# 🚀 Универсальная инициализация окружения и данных
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
PRIMARY = "#1779ba"
SUCCESS = "#3adb76"
ALERT = "#cc4b37"
WARNING = "#ffae00"

# Генерация синтетического лога активности пользователей (логины по дням)
np.random.seed(42)
users = [f"U_{i:03d}" for i in range(1, 21)]
dates = pd.date_range("2027-01-01", periods=15, freq="D")

records = []
for u in users:
    active_days = np.random.choice(dates, size=np.random.randint(4, 12), replace=False)
    for d in sorted(active_days):
        records.append({"user_id": u, "login_date": d})

logins_df = pd.DataFrame(records).sort_values(["user_id", "login_date"]).reset_index(drop=True)
print(f"Всего событий активности: {len(logins_df)} записей")
display(logins_df.head(8))

## 🔍 Разбор механики: Алгоритм Gaps & Islands в Pandas
Классический вопрос live-coding: *«Найдите максимальное количество дней непрерывной активности подряд (Max Streak) для каждого пользователя»*.
Вместо медленного цикла `for` используется векторный трюк:
$$\text{Island Group} = \text{cumsum}(\text{date} - \text{lag}(\text{date}) > 1 \text{ day})$$

In [ ]:
# Решение live-coding задачи: Острова непрерывной активности (Streak)
df = logins_df.copy()
# 1. Вычисляем разницу между текущей и предыдущей датой логина пользователя
df['prev_date'] = df.groupby('user_id')['login_date'].shift(1)
df['day_diff'] = (df['login_date'] - df['prev_date']).dt.days

# 2. Если разрыв > 1 дня (или первый вход), создаем маркер нового острова
df['new_island'] = (df['day_diff'] != 1).astype(int)
df['island_id'] = df.groupby('user_id')['new_island'].cumsum()

# 3. Длина каждого непрерывного стрика
streaks = df.groupby(['user_id', 'island_id']).size().reset_index(name='streak_days')
max_streaks = streaks.groupby('user_id')['streak_days'].max().reset_index(name='max_streak')

print("Топ-5 пользователей по максимальному стрику логинов:")
display(max_streaks.sort_values('max_streak', ascending=False).head(5))

## 📊 Визуализация бизнес-кейса: Распределение стриков удержания
Построим распределение стриков для выявления когорты самых вовлеченных пользователей (Power Users).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5), dpi=110)

streak_counts = max_streaks['max_streak'].value_counts().sort_index()
bars = ax.bar(streak_counts.index, streak_counts.values, color=PRIMARY, edgecolor='#333333', linewidth=0.6)

ax.set_title("Распределение кандидатов в когорту Power Users по максимальному стрику", fontsize=12, fontweight='bold', pad=15)
ax.set_xlabel("Длина непрерывной активности (дней подряд)", fontsize=11)
ax.set_ylabel("Количество пользователей", fontsize=11)

for b in bars:
    h = b.get_height()
    ax.annotate(f"{h}", xy=(b.get_x() + b.get_width()/2, h), xytext=(0, 4),
                textcoords="offset points", ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

### 🎯 Практическое задание 1: Time-to-Second-Order
Рассчитайте среднее время в днях от 1-го до 2-го логина для пользователей, у которых было как минимум 2 дня активности.

In [ ]:
### ВАШ КОД ЗДЕСЬ
first_two = logins_df.groupby('user_id').head(2)
user_counts = first_two.groupby('user_id').size()
valid_users = user_counts[user_counts >= 2].index

piv = first_two[first_two['user_id'].isin(valid_users)].copy()
piv['order_num'] = piv.groupby('user_id').cumcount() + 1
t2o = piv.pivot(index='user_id', columns='order_num', values='login_date')
diff_days = (t2o[2] - t2o[1]).dt.days

mean_days_to_second = diff_days.mean()
print(f"Среднее время до 2-го входа: {mean_days_to_second:.2f} дней")

assert mean_days_to_second > 0, "Интервал должен быть положительным числом"
print("✅ Тест успешно пройден: алгоритм расчета retention-метрик векторизован безошибочно!")

### 💡 Совет по прохождению Live-Coding
На собеседовании всегда сначала проговаривайте решение вслух ($O(N)$ по времени и памяти), проверяйте граничные случаи (1 день активности, пропуски), и только затем пишите векторизованный код.